# 第14回 変化点と異常の検知

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：「いつもと違う」を、データからどう見つけるか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "tsibbledata", "fable", "feasts", "changepoint"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

In [ ]:
library(tsibble)
library(tsibbledata)
library(feasts)
library(fable)
library(ggtime)   # ACFと分解の図（autoplot）
library(dplyr)
library(ggplot2)
options(repr.plot.width = 8, repr.plot.height = 4.5)
invisible(Sys.setlocale("LC_TIME", "C"))   # 図の月の名前を英語にする

`changepoint`パッケージは、`library()`で読み込まず、`changepoint::`を付けて呼びます。

### A-1. 変化点と異常

「いつもと違う」には、2種類あります。

- 変化点（change point）：ある時点を境に、系列の性質（平均や分散）が変わり、変わったまま戻らない。
- 異常（anomaly）：その時点だけ、ふつうの動きから大きく外れる。次の時点には戻る。

同じノイズから、2本の系列を作ります。1本目は121時点目から平均が2だけ上がります。2本目は121時点目だけ6だけ上がります。

In [ ]:
set.seed(1)
n <- 200
noise <- rnorm(n)                                       # ノイズ（標準偏差1）
x_shift <- noise + ifelse(1:n > 120, 2, 0)              # 変化点：121時点目から平均が2上がる
x_spike <- replace(noise, 121, noise[121] + 6)          # 異常：121時点目だけ6上がる
kinds <- data.frame(t = rep(1:n, 2),
                    kind = rep(c("1 change point (the mean stays shifted)",
                                 "2 anomaly (only one point is off)"), each = n),
                    x = c(x_shift, x_spike))
ggplot(kinds, aes(t, x)) +
  geom_line() +
  facet_wrap(~kind, ncol = 1) +
  labs(x = "Time t (step)", y = "Value x_t")

### A-2. 変化点が1つ、平均だけが変わる

系列を時点$\tau$で2つに分け、それぞれの区間の平均を引きます。残差の2乗和を$\mathrm{SSE}(\tau)$と書きます。

$$
\mathrm{SSE}(\tau) = \sum_{t=1}^{\tau} \left( x_t - \bar{x}_{1:\tau} \right)^2 + \sum_{t=\tau+1}^{n} \left( x_t - \bar{x}_{\tau+1:n} \right)^2
$$

$\tau$は「変わる直前の時点」です。$\mathrm{SSE}(\tau)$が最も小さくなる$\tau$を、変化点の推定値$\hat{\tau}$とします。

In [ ]:
# すべての分け方 τ = 1, ..., n-1 について、SSE(τ) を返す
sse_split <- function(x) {
  n <- length(x); tau <- 1:(n - 1)
  s <- cumsum(x); s2 <- cumsum(x^2)                     # 和と、2乗の和
  left  <- s2[tau] - s[tau]^2 / tau                     # 左の区間の残差の2乗和
  right <- (s2[n] - s2[tau]) - (s[n] - s[tau])^2 / (n - tau)   # 右の区間
  left + right
}

In [ ]:
sse <- sse_split(x_shift)
tau_hat <- which.min(sse)
c(tau_hat = tau_hat,
  sse_no_split = sum((x_shift - mean(x_shift))^2),      # 分けないときの残差の2乗和
  sse_best = min(sse))                                  # 最も良い分け方での残差の2乗和
ggplot(data.frame(tau = 1:(n - 1), sse = sse), aes(tau, sse)) +
  geom_line() +
  geom_vline(xintercept = tau_hat, colour = "red", linetype = "dashed") +
  labs(x = "Split point tau (step)", y = "SSE(tau)")

本当の変化点は120です。$\mathrm{SSE}(\tau)$は、その近くで深い谷になります。

1本だけでは、たまたまかもしれません。平均の変わる幅を0.5、1、2と変え、1000本ずつ作って、$\hat{\tau}$が本当の変化点（120）からどのくらいずれるかを調べます。

In [ ]:
set.seed(1)
res_one <- do.call(rbind, lapply(c(0.5, 1, 2), function(delta) {
  tau <- replicate(1000, which.min(sse_split(rnorm(200) + ifelse(1:200 > 120, delta, 0))))
  data.frame(delta = delta,
             exact = mean(tau == 120),                  # ちょうど当たった割合
             within5 = mean(abs(tau - 120) <= 5),       # 前後5時点以内に入った割合
             sd_tau = sd(tau))                          # 推定値のばらつき
}))
res_one

- 変わる幅が大きいほど、変化点の場所を正確に当てられます。
- 幅が0.5（ノイズの標準偏差の半分）だと、前後5時点以内に入るのは半分以下です。

次に、変化点のない系列（平均0、標準偏差1のiid）で同じことをします。
分けないときと、最も良い分け方のときとで、残差の2乗和がどれだけ減るかを見ます。これを`gain`と呼びます。

In [ ]:
set.seed(1)
gain <- replicate(2000, {
  x <- rnorm(200)
  sum((x - mean(x))^2) - min(sse_split(x))
})
round(quantile(gain, c(0.5, 0.9, 0.95, 0.99)), 2)
c(over_2 = mean(gain > 2), over_2logn = mean(gain > 2 * log(200)))
ggplot(data.frame(gain = gain), aes(gain)) +
  geom_histogram(binwidth = 0.5, boundary = 0, fill = "grey60") +
  geom_vline(xintercept = c(2, 2 * log(200)), colour = "red", linetype = "dashed") +
  labs(x = "Reduction in SSE by the best split (no change point in truth)", y = "Count (out of 2000)")

- 変化点がなくても、分ければ残差の2乗和は必ず減ります（`gain`は必ず正）。
- 「2より多く減ったら変化点あり」とすると、変化点のない系列のほとんどで「あり」と答えてしまいます。
- 「$2 \log n$（$n = 200$で約10.6）より多く減ったら」とすれば、誤りは数%で済みます。

「減った」だけでは、変化点があるとは言えません。どのくらい減れば認めるかの基準が要ります。

### A-3. 変化点がいくつあるか分からない

変化点を$m$個置くと、系列は$m + 1$個の区間に分かれます。区間ごとの平均を引いた残差の2乗和の合計を$\mathrm{SSE}$とします。
変化点を増やせば、$\mathrm{SSE}$は必ず減ります。そこで、変化点の数に罰則（penalty）を付けます。

$$
\frac{\mathrm{SSE}}{\sigma^2} + \beta\, m
$$

これが最も小さくなる変化点の置き方を選びます。$\sigma$はノイズの標準偏差、$\beta$は変化点1つあたりの罰則です。
第3回のAIC・BICと同じ、当てはまりと複雑さのつり合いです。今日は$\beta = 2 \log n$を基本にし、その何倍にするかを`mult`で決めます。

$\sigma$は分からないので、隣どうしの差から見積もります。変化点でない場所では、差$x_t - x_{t-1}$はノイズ2個の差なので、分散は$2\sigma^2$です。

すべての置き方を調べると時間がかかるので、`changepoint`パッケージのPELT法（Pruned Exact Linear Time）で探します。

In [ ]:
# ノイズの標準偏差を、隣どうしの差から見積もる
noise_sd <- function(x) sd(diff(x)) / sqrt(2)
# 平均の変化点を探す。mult は罰則の強さ（1 のとき、変化点1つにつき 2 log n）
find_cp <- function(x, mult = 1) {
  z <- x / noise_sd(x)                                  # ノイズの標準偏差が1になるように割る
  fit <- changepoint::cpt.mean(z, method = "PELT", penalty = "Manual",
                               pen.value = mult * 2 * log(length(x)))
  changepoint::cpts(fit)                                # 各区間の最後の時点
}
# 時点ごとに、何番目の区間にいるか
seg_id <- function(n, cp) findInterval(1:n, cp + 1) + 1
# 区間ごとの平均
seg_mean <- function(x, cp) ave(x, seg_id(length(x), cp))
# 系列に、変化点（赤い破線）と区間ごとの平均（赤い線）を重ねる
plot_cp <- function(time, x, cp) {
  ggplot(data.frame(time = time, x = x, m = seg_mean(x, cp), seg = seg_id(length(x), cp))) +
    geom_line(aes(time, x), colour = "grey50") +
    geom_line(aes(time, m, group = seg), colour = "red", linewidth = 1) +
    geom_vline(xintercept = time[cp], colour = "red", linetype = "dashed")
}

変化点が3つ（80、200、260）ある、長さ400の系列を作ります。平均は0 → 1.5 → −0.5 → 1と変わります。

In [ ]:
set.seed(1)
n4 <- 400
truth <- c(80, 200, 260)                                # 本当の変化点
mu4 <- rep(c(0, 1.5, -0.5, 1), diff(c(0, truth, n4)))   # 区間ごとの平均
x4 <- mu4 + rnorm(n4)
cp4 <- find_cp(x4, mult = 1)
cp4
plot_cp(1:n4, x4, cp4) +
  labs(x = "Time t (step)", y = "Value x_t")

罰則の強さ`mult`を変えると、見つかる変化点の数が変わります。

In [ ]:
path <- do.call(rbind, lapply(2^seq(-4, 4, 0.5), function(m) {
  cp <- find_cp(x4, m)
  data.frame(mult = m, n_cp = length(cp), sse = sum((x4 - seg_mean(x4, cp))^2))
}))
round(path[path$mult %in% c(0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16), ], 2)
ggplot(path, aes(mult, n_cp)) +
  geom_step() + geom_point() +
  geom_hline(yintercept = 3, colour = "red", linetype = "dashed") +
  scale_x_log10() +
  labs(x = "Penalty multiplier mult (log scale)", y = "Number of change points found")

- 罰則が弱いと、変化点をいくらでも見つけます。そのぶん`sse`は小さくなります。
- 罰則を強くしていくと、本当の数（3）で長く止まり、さらに強くすると1つも見つからなくなります。

見つけた変化点を、答えと比べる関数を用意します。答えを知っている状態で試すのは、第2回と同じ考え方です。

- `hit`：本当の変化点のうち、前後5時点以内に見つけたものがある割合
- `false`：見つけた変化点のうち、どの本当の変化点からも5時点より離れているものの数

In [ ]:
score_cp <- function(found, truth, tol = 5) {
  hit <- sapply(truth, function(t) any(abs(found - t) <= tol))
  false <- sum(vapply(found, function(f) all(abs(truth - f) > tol), logical(1)))
  c(hit = mean(hit), false = false)
}

同じ形の系列を200本作り、罰則の強さごとに平均します。

In [ ]:
set.seed(1)
sims <- replicate(200, mu4 + rnorm(n4), simplify = FALSE)   # 200本（どの mult でも同じ200本を使う）
res_cp <- do.call(rbind, lapply(c(0.25, 0.5, 1, 2, 4, 8), function(m) {
  s <- rowMeans(sapply(sims, function(x) score_cp(find_cp(x, m), truth)))
  data.frame(mult = m, hit = s[["hit"]], false = s[["false"]])
}))
round(res_cp, 3)

- 罰則が弱い（0.25）と、誤って拾う数が多くなります。
- 罰則が強すぎる（8）と、本当の変化点も見つけられません。
- そのあいだ（1〜4）では、ほとんどの変化点を見つけ、誤って拾う数も少なくなります。

### A-4. 分散が変わる

平均は変わらず、ばらつきだけが変わることもあります。251時点目から、標準偏差が1から3になる系列を作ります。

区間ごとの当てはまりの悪さを、残差の2乗和の代わりに、「区間の長さ × log(区間の分散)」で測ります。考え方は平均のときと同じです。

In [ ]:
# 分散の変化点を探す（平均は系列全体で一定とする）
find_cp_var <- function(x, mult = 1) {
  fit <- changepoint::cpt.var(x - mean(x), method = "PELT", penalty = "Manual",
                              pen.value = mult * 2 * log(length(x)),
                              know.mean = TRUE, mu = 0)
  changepoint::cpts(fit)
}

In [ ]:
set.seed(1)
x_var <- c(rnorm(250, sd = 1), rnorm(150, sd = 3))
cp_var <- find_cp_var(x_var)
list(variance = cp_var, mean = find_cp(x_var))          # 分散の変化点と、平均の変化点
tapply(x_var, seg_id(400, cp_var), sd)                  # 区間ごとの標準偏差
ggplot(data.frame(t = 1:400, x = x_var), aes(t, x)) +
  geom_line() +
  geom_vline(xintercept = cp_var, colour = "red", linetype = "dashed") +
  labs(x = "Time t (step)", y = "Value x_t")

分散の変化点は、本当の場所（250）の近くに見つかります。平均の変化点は、見つかりません。

### A-5. 第1回とのつながり：定常でない系列

平均が途中で変わる系列は、定常ではありません。A-3の系列`x4`のACFを、2通りに出します。

- `1 raw`：系列そのまま
- `2 segment means removed`：区間ごとの平均を引いたもの

In [ ]:
acf_sim <- tsibble(t = rep(1:n4, 2),
                   series = rep(c("1 raw", "2 segment means removed"), each = n4),
                   x = c(x4, x4 - seg_mean(x4, cp4)),
                   key = series, index = t)
acf_sim |>
  ACF(x, lag_max = 30) |>
  autoplot() +
  labs(x = "Lag h (step)", y = "Sample ACF")

- そのままのACFは、ゆっくりとしか小さくなりません。ノイズはiidなのに、強い自己相関があるように見えます。
- 区間ごとの平均を引くと、ほぼすべてのラグで破線の中に入ります。区間の中は、定常として扱えます。

第4回では、定常でない系列を差分で定常にしました。変化点で分けるのは、それとは別のやり方です。

### A-6. 自己相関があると、変化点を見つけすぎる

ここまでの方法は、区間の中ではiid（独立で同じ分布）と仮定しています。
変化点のないAR(1)過程（$\phi = 0.7$）に、同じ方法を使います。

In [ ]:
set.seed(1)
sim_ar1 <- function(n, phi) {           # AR(1)を1本作る（分散が1になるように縮める）
  x <- stats::filter(rnorm(n + 100), phi, method = "recursive")
  as.numeric(x[-(1:100)]) * sqrt(1 - phi^2)
}
x_ar <- sim_ar1(400, 0.7)
cp_ar <- find_cp(x_ar)
length(cp_ar)
plot_cp(1:400, x_ar, cp_ar) +
  labs(x = "Time t (step)", y = "Value x_t (AR(1), no change point)")

変化点は1つもないのに、たくさん見つかります。AR(1)は、しばらく高いところや低いところにとどまります。それを、平均が変わったと読んでしまうからです。

$\phi$と罰則の強さを変えて、200本ずつ作り、誤って見つけた変化点の数を平均します。

In [ ]:
set.seed(1)
res_ar <- do.call(rbind, lapply(c(0, 0.4, 0.7), function(phi) {
  sims_ar <- replicate(200, sim_ar1(400, phi), simplify = FALSE)
  n_found <- sapply(c(1, 2, 4, 8), function(m) mean(sapply(sims_ar, function(x) length(find_cp(x, m)))))
  data.frame(phi = phi, mult_1 = n_found[1], mult_2 = n_found[2], mult_4 = n_found[3], mult_8 = n_found[4])
}))
round(res_ar, 2)

- $\phi = 0$（iid）なら、誤って見つける数はほぼ0です。
- $\phi$が大きいほど、誤って見つける数が増えます。罰則を強くすれば減ります。

対処を2つ試します。A-3の系列の平均（変化点3つ）に、iidのノイズの代わりにAR(1)のノイズを足した系列を使います。

- `strong`：罰則を強くする。$\phi$が分かっているとして、`mult`を$(1 + \phi)/(1 - \phi)^2$倍にする（式の出どころは読み物）。
- `resid`：先にAR(1)モデルを当てはめ、その残差（1時点先の予測の誤差）に対して変化点を探す。

In [ ]:
set.seed(1)
res_remedy <- do.call(rbind, lapply(c(0.4, 0.7), function(phi) {
  sims_ar <- replicate(200, mu4 + sim_ar1(n4, phi), simplify = FALSE)
  strong <- (1 + phi) / (1 - phi)^2
  s <- cbind(
    plain  = rowMeans(sapply(sims_ar, function(x) score_cp(find_cp(x, 1), truth))),
    strong = rowMeans(sapply(sims_ar, function(x) score_cp(find_cp(x, strong), truth))),
    resid  = rowMeans(sapply(sims_ar, function(x) {
      e <- as.numeric(residuals(arima(x, order = c(1, 0, 0))))
      score_cp(find_cp(e, 1), truth)
    })))
  data.frame(phi = phi, method = colnames(s), hit = s["hit", ], false = s["false", ], row.names = NULL)
}))
res_remedy$hit <- round(res_remedy$hit, 2); res_remedy$false <- round(res_remedy$false, 2)
res_remedy

- `plain`（何もしない）：変化点は見つけますが、誤って拾う数が多すぎます。
- `strong`：誤って拾う数は減ります。$\phi = 0.4$では、見つける割合は変わりません。$\phi = 0.7$では、本当の変化点の半分ほどを見落とします。
- `resid`：誤って拾う数は減りますが、本当の変化点も見落とします。$\phi = 0.7$では、ほとんど見つけられません。平均の変化を、AR(1)の係数が「強い自己相関」として吸い取ってしまうからです。

どちらの対処にも代償があります。ゆっくり動く自己相関と、平均の変化は、データだけからは見分けにくいのです。

### A-7. 異常を拾う：分解の残り

異常検知は、次の2段階で行います。

1. 「ふつうの動き」をモデルで表し、系列から引く。
2. 引いた残りが、「ふつうの幅」の$k$倍より外にある時点を拾う。

「ふつうの幅」の測り方を3つ用意します。残りを$r_t$とします。

- `sd`：平均と標準偏差。
- `mad`：中央値と、中央絶対偏差（median absolute deviation, MAD）。$\mathrm{MAD} = \mathrm{median}\,|r_t - \mathrm{median}(r)|$。1.4826倍すると、正規分布のときに標準偏差と同じ大きさになります。
- `iqr`：中央値と、四分位範囲（interquartile range, IQR）。第3四分位数から第1四分位数を引いたもの。1.349で割ると、正規分布のときに標準偏差と同じ大きさになります。

In [ ]:
# 「ふつうの幅」の測り方（3通り）。中心（centre）と幅（width）を返す
spread <- function(r, type = "mad") {
  switch(type,
    sd  = c(centre = mean(r),   width = sd(r)),
    mad = c(centre = median(r), width = 1.4826 * median(abs(r - median(r)))),
    iqr = c(centre = median(r), width = IQR(r) / 1.349))
}
# 中心から、幅の k 倍より離れた時点を拾う（TRUE が異常）
flag <- function(r, k = 3, type = "mad") {
  s <- spread(r, type)
  abs(r - s[["centre"]]) > k * s[["width"]]
}
# 残りに、基準の線（青い破線）と、拾った時点（赤い点）を重ねる
plot_flag <- function(time, r, k = 3, type = "mad") {
  s <- spread(r, type); f <- flag(r, k, type)
  ggplot(data.frame(time = time, r = r), aes(time, r)) +
    geom_line(colour = "grey50") +
    geom_hline(yintercept = s[["centre"]] + c(-1, 1) * k * s[["width"]], colour = "blue", linetype = "dashed") +
    geom_point(data = data.frame(time = time[f], r = r[f]), colour = "red", size = 2)
}
# 拾った時点を、答えと比べる（hit：見つけた割合、false：誤って拾った数）
score_an <- function(found, truth) c(hit = mean(truth %in% found), false = sum(!(found %in% truth)))

第2回と同じ形の系列（傾向 ＋ 曜日ごとの上下 ＋ 標準偏差2のノイズ、364日）を作り、8日だけ、値を10（ノイズの標準偏差の5倍）だけ上か下にずらします。

In [ ]:
set.seed(1)
n <- 364
day <- 1:n
trend  <- 100 + 8 * sin(2 * pi * day / 364) + 0.02 * day    # なめらかな傾向
season <- rep(c(5, 6, 6, 5, 4, -12, -14), length.out = n)   # 7日でくり返す上下
make_anom <- function(m, size) {        # m 日だけ、size だけ上か下にずらした系列を作る
  at <- sort(sample(10:(n - 10), m))                        # ずらす日
  y <- trend + season + rnorm(n, sd = 2)
  y[at] <- y[at] + size * sample(c(-1, 1), m, replace = TRUE)
  list(y = y, at = at)
}
d <- make_anom(m = 8, size = 10)
d$at
ggplot(data.frame(day = day, y = d$y), aes(day, y)) +
  geom_line() +
  geom_point(data = data.frame(day = d$at, y = d$y[d$at]), colour = "red", size = 2) +
  labs(x = "Day", y = "Value (red: embedded anomalies)")

曜日ごとの上下が大きいので、系列を見ただけでは、ずらした日（赤い点）は目立ちません。

「ふつうの動き」を、第2回の分解（傾向 ＋ 季節）で表します。STL分解の残りを取り出します。

In [ ]:
dcmp_sim <- tsibble(day = day, y = d$y, index = day) |>
  model(STL(y ~ season(period = 7, window = "periodic"))) |>
  components()
r_sim <- dcmp_sim$remainder
plot_flag(day, r_sim, k = 3, type = "mad") +
  labs(x = "Day", y = "Remainder (red: flagged, dashed: median +/- 3 width)")
score_an(which(flag(r_sim, 3, "mad")), d$at)

残りでは、ずらした日がとびぬけて見えます。

幅の測り方と$k$を変えて、拾った数、見つけた割合、誤って拾った数を比べます。

In [ ]:
round(sapply(c("sd", "mad", "iqr"), function(ty) spread(r_sim, ty)), 2)
rules <- expand.grid(k = c(2, 3, 4), type = c("sd", "mad", "iqr"), stringsAsFactors = FALSE)
res_rule <- cbind(rules, t(mapply(function(k, ty) {
  found <- which(flag(r_sim, k, ty))
  c(n_flag = length(found), score_an(found, d$at))
}, rules$k, rules$type)))
res_rule

- $k$を小さくすると、拾う数が増えます。$k = 2$では、ずらしていない日もたくさん拾います。
- 標準偏差（`sd`）は、ほかの2つより大きく出ます。ずらした8日が、標準偏差を大きくしているからです。

異常が増えると、この違いが大きくなります。異常の数を8日と30日、ずらす幅を12にして、200本ずつ作り、$k = 3$で比べます。
ここでは、STL分解を速く計算するために`stats::stl()`を使います（上の`STL()`と同じ分解です）。

In [ ]:
set.seed(1)
stl_remainder <- function(y) as.numeric(stl(ts(y, frequency = 7), s.window = "periodic")$time.series[, "remainder"])
res_study <- do.call(rbind, lapply(c(8, 30), function(m) {
  sims_an <- replicate(200, make_anom(m, size = 12), simplify = FALSE)
  do.call(rbind, lapply(c("sd", "mad", "iqr"), function(ty) {
    s <- sapply(sims_an, function(d) {
      r <- stl_remainder(d$y)
      c(width = spread(r, ty)[["width"]], score_an(which(flag(r, 3, ty)), d$at))
    })
    data.frame(m = m, type = ty, width = mean(s["width", ]), hit = mean(s["hit", ]), false = mean(s["false", ]))
  }))
}))
res_study[, 3:5] <- round(res_study[, 3:5], 2)
res_study

- 異常が8日なら、標準偏差の基準でも、ほとんどの異常を見つけます。
- 異常が30日あると、標準偏差はノイズの本当の標準偏差（2）よりずっと大きくなります。基準の線が外に広がり、異常の多くを見落とします。
- MADとIQRは、異常が増えてもあまり変わりません。少数の極端な値に引きずられにくい、という性質を、頑健（robust）といいます。

### A-8. 異常を拾う：1時点先の予測の誤差

「ふつうの動き」を、1時点先の予測で表すこともできます。昨日までの値から今日の値を予測し、予測の誤差が大きい時点を拾います。

AR(1)過程（$\phi = 0.7$、長さ300）を作り、4時点だけ、値を8だけ上か下にずらします。AR(1)モデルを当てはめ、1時点先の予測の誤差を出します。

In [ ]:
set.seed(1)
at_ar <- c(60, 130, 200, 250)                               # ずらす時点
y_ar <- as.numeric(arima.sim(list(ar = 0.7), 300))
y_ar[at_ar] <- y_ar[at_ar] + c(8, -8, 8, -8)
fit_ar <- tsibble(t = 1:300, y = y_ar, index = t) |>
  model(ar1 = ARIMA(y ~ 1 + pdq(1, 0, 0)))
e_ar <- augment(fit_ar)$.innov                              # 1時点先の予測の誤差
plot_flag(1:300, e_ar, k = 3, type = "mad") +
  labs(x = "Time t (step)", y = "One-step forecast error (red: flagged)")
which(flag(e_ar, 3, "mad"))
data.frame(t = at_ar, error_at_t = round(e_ar[at_ar], 2), error_at_next = round(e_ar[at_ar + 1], 2))
c(phi_hat = tidy(fit_ar)$estimate[1])

- ずらした時点の誤差は、ずらした幅（±8）に近い値になります。
- 次の時点の誤差も、逆向きに大きくなります。次の時点の予測は、ずれた値をもとに作られるからです。
- 分解の残りは、前後の両方の値を使って作ります。予測の誤差は、過去の値だけで作ります。そのため、新しい値が届くたびに判定できますが、異常の「こだま」が次の時点に出ます。
- 係数の推定値`phi_hat`は、作ったときの0.7より小さくなります。異常は、モデルの推定もゆがめます。

## B. 実データを解析する

2つのデータを使います。

- 変化点：Rに付属する年ごとの系列（最初の設定は`Nile`。ナイル川の年間流量）
- 異常：`tsibbledata`パッケージの`vic_elec`（オーストラリアのビクトリア州の電力需要。第2回と同じ）

B節は、A節で作った関数（`sse_split`、`find_cp`、`flag`など）を使います。A節を実行してから進んでください。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
cp_series  <- "Nile"     # 変化点を探す系列："Nile"（ナイル川の流量） "LakeHuron"（ヒューロン湖の水位） "nhtemp"（ニューヘイブンの気温）
pen_mult   <- 1          # 罰則の強さ（変化点1つにつき 2 log n の何倍か）：0.5〜4
series     <- "Demand"   # 異常を探す系列："Demand"（1日の電力需要）か "Temperature"（1日の最高気温）
year_from  <- 2014       # 始まりの年：2012〜2014
year_to    <- 2014       # 終わりの年：year_from 以上、2014 以下
width_type <- "mad"      # ふつうの幅の測り方："sd" "mad" "iqr"
k          <- 3          # 幅の何倍より外を拾うか：2〜5

### B-1. 変化点：データを確かめる

In [ ]:
cp_ts <- switch(cp_series, Nile = Nile, LakeHuron = LakeHuron, nhtemp = nhtemp)
x_lab <- switch(cp_series,
                Nile = "Annual flow (10^8 m^3)",
                LakeHuron = "Lake level (feet)",
                nhtemp = "Mean annual temperature (deg F)")
cp_data <- as_tsibble(cp_ts)
glimpse(cp_data)
interval(cp_data)
x  <- as.numeric(cp_ts)        # 値
yr <- as.numeric(time(cp_ts))  # 年
c(from = min(yr), to = max(yr), n = length(x), missing = sum(is.na(x)))
ggplot(cp_data, aes(index, value)) +
  geom_line() +
  labs(x = "Year", y = x_lab)

- `Nile`は、アスワンで測ったナイル川の年間流量（$10^8\,\mathrm{m}^3$）です。1871年から1970年までの100年分で、1行が1年です。欠測はありません。

### B-2. 変化点を1つだけ探す

A-2と同じように、系列を1か所で2つに分け、残差の2乗和が最も小さくなる年を探します。

In [ ]:
sse_real <- sse_split(x)
tau_real <- which.min(sse_real)
c(last_year_before_change = yr[tau_real],
  mean_before = mean(x[1:tau_real]), mean_after = mean(x[-(1:tau_real)]))
ggplot(data.frame(year = yr[-length(yr)], sse = sse_real), aes(year, sse)) +
  geom_line() +
  geom_vline(xintercept = yr[tau_real], colour = "red", linetype = "dashed") +
  labs(x = "Split point (last year of the first segment)", y = "SSE")

### B-3. 変化点の数を、罰則で決める

In [ ]:
cp <- find_cp(x, pen_mult)
yr[cp]                                                  # 各区間の最後の年
seg_table <- data.frame(year = yr, x = x, seg = seg_id(length(x), cp)) |>
  group_by(seg) |>
  summarise(from = min(year), to = max(year), years = n(), mean = mean(x), sd = sd(x))
seg_table
plot_cp(yr, x, cp) +
  labs(x = "Year", y = x_lab)

罰則の強さを変えると、見つかる変化点の数がどう変わるかを見ます。

In [ ]:
path_real <- data.frame(mult = c(0.25, 0.5, 1, 2, 4, 8))
path_real$n_cp <- sapply(path_real$mult, function(m) length(find_cp(x, m)))
path_real

### B-4. 分けたあとは、定常として扱えるか

系列そのままと、区間ごとの平均を引いたものとで、ACFを比べます。

In [ ]:
n_x <- length(x)
acf_data <- tsibble(t = rep(1:n_x, 2),
                    series = rep(c("1 raw", "2 segment means removed"), each = n_x),
                    x = c(x, x - seg_mean(x, cp)),
                    key = series, index = t)
acf_real <- acf_data |> ACF(x, lag_max = 20)
acf_real |>
  autoplot() +
  labs(x = "Lag h (year)", y = "Sample ACF")
acf_real |> as_tibble() |> filter(lag <= 3) |> mutate(acf = round(acf, 2))

読み方：

- B-2、B-3：`Nile`では、1898年を最後に、流量の平均が下がります。第9回で見た「1899年の落差」と同じ変化です（ここでは、変わる直前の年を変化点と呼んでいます）。罰則を1より強くしても、変化点は1つのままです。
- B-4：そのままのACFは、ゆっくり小さくなります。区間ごとの平均を引くと、ほぼ破線の中に入ります。A-5と同じです。

### B-5. 異常：データを確かめ、前処理をする

In [ ]:
glimpse(vic_elec)
interval(vic_elec)
range(vic_elec$Date)
c(rows = nrow(vic_elec), missing = sum(is.na(vic_elec$Demand)), gaps = has_gaps(vic_elec)$.gaps)

- 1行は30分です。`Demand`が電力需要（MWh）、`Temperature`がメルボルンの気温（℃）、`Holiday`が祝日の印です。
- 期間は2012年1月1日から2014年12月31日、52608行です。欠測も、行の抜けもありません。

第2回と同じように、1日ごとにまとめます。電力需要は1日の合計（GWh）、気温は1日の最高値にします。

In [ ]:
daily_all <- vic_elec |>
  index_by(Date) |>
  summarise(Demand = sum(Demand) / 1000,          # 1日の電力需要（GWh）
            Temperature = max(Temperature),       # 1日の最高気温（℃）
            Holiday = any(Holiday))               # 祝日かどうか
daily <- daily_all |>
  filter(format(Date, "%Y") >= year_from, format(Date, "%Y") <= year_to) |>
  mutate(y = .data[[series]])
y_lab <- if (series == "Demand") "Daily demand (GWh)" else "Daily max temperature (deg C)"
interval(daily)
n <- nrow(daily)
n
ggplot(daily, aes(Date, y)) +
  geom_line() +
  labs(x = "Date", y = y_lab)

### B-6. ふつうの動き①：分解

In [ ]:
dcmp <- daily |>
  model(STL(y ~ season(period = 7, window = "periodic"))) |>
  components()
autoplot(dcmp) +
  labs(x = "Date")

残りが、設定した基準（`width_type`の幅の`k`倍）より外にある日を拾います。

In [ ]:
r <- dcmp$remainder
flag_stl <- flag(r, k, width_type)
round(spread(r, width_type), 2)
sum(flag_stl)
plot_flag(daily$Date, r, k, width_type) +
  labs(x = "Date", y = "Remainder (red: flagged)")

拾った日が、何の日なのかを読みます。曜日、気温、祝日の印を並べます。

In [ ]:
flag_table <- daily |>
  as_tibble() |>
  mutate(wday = format(Date, "%a"), remainder = round(r, 1)) |>
  filter(flag_stl) |>
  select(Date, wday, Demand, Temperature, Holiday, remainder)
print(flag_table, n = 40)
flag_table |>
  summarise(days = n(), holidays = sum(Holiday), hot_35 = sum(Temperature >= 35),
            first_two_months = sum(format(Date, "%m") %in% c("01", "02")))

幅の測り方と$k$を変えると、拾う日数がどう変わるかを見ます。

In [ ]:
round(sapply(c("sd", "mad", "iqr"), function(ty) spread(r, ty)), 2)
n_flag <- sapply(c("sd", "mad", "iqr"), function(ty) sapply(2:5, function(kk) sum(flag(r, kk, ty))))
rownames(n_flag) <- paste0("k=", 2:5)
n_flag

### B-7. ふつうの動き②：1時点先の予測

第4回の季節ARIMAモデルを当てはめ、1時点先の予測の誤差を出します。
モデルは、「1週間前の同じ曜日との差が、ARMA(1, 1)に従う」という形に決めておきます（ARIMA(1, 0, 1)(0, 1, 1)$_7$）。

In [ ]:
fit <- daily |>
  model(sarima = ARIMA(y ~ 0 + pdq(1, 0, 1) + PDQ(0, 1, 1, period = 7)))
e <- augment(fit)$.innov                                # 1時点先の予測の誤差
flag_pred <- flag(e, k, width_type)
plot_flag(daily$Date, e, k, width_type) +
  labs(x = "Date", y = "One-step forecast error (red: flagged)")
table(decomposition = flag_stl, forecast = flag_pred)   # 2つの方法で拾った日の重なり

予測の誤差でだけ拾った日を、前の日の様子と並べます。

In [ ]:
pred_table <- daily |>
  as_tibble() |>
  mutate(wday = format(Date, "%a"), error = round(e, 1),
         prev_flag = lag(flag_pred, default = FALSE),   # 前の日も拾われているか
         prev_holiday = lag(Holiday, default = FALSE)) |>   # 前の日が祝日か
  filter(flag_pred, !flag_stl) |>
  select(Date, wday, Temperature, Holiday, error, prev_flag, prev_holiday)
print(pred_table, n = 40)

### B-8. 残りの幅は、一年じゅう同じか

残りに対して、A-4の分散の変化点を探します。

In [ ]:
cp_v <- find_cp_var(r, pen_mult)
var_table <- data.frame(Date = daily$Date, r = r, flagged = flag_stl, seg = seg_id(n, cp_v)) |>
  group_by(seg) |>
  summarise(from = min(Date), to = max(Date), days = n(),
            width = spread(r, "mad")[["width"]],        # 区間ごとの幅（MAD）
            flagged = sum(flagged))                     # B-6で拾った日の数
var_table
ggplot(data.frame(Date = daily$Date, r = r), aes(Date, r)) +
  geom_line(colour = "grey50") +
  geom_vline(xintercept = daily$Date[cp_v], colour = "red", linetype = "dashed") +
  labs(x = "Date", y = "Remainder (dashed: change points in variance)")

読み方：

- B-6：拾った日は、猛暑の日（需要が上に外れる）と、祝日（需要が下に外れる）が中心です。猛暑のすぐ前後の、ふつうの日も拾われます。猛暑が傾向を引き上げ、まわりの日が「いつもより低い」ことになるからです。
- B-6：標準偏差は、MADやIQRよりずっと大きく出ます。A-7で見たとおり、外れた日が標準偏差を大きくしています。
- B-7：予測の誤差でだけ拾った日には、祝日の次の日（4月19日）や、猛暑の次の日（1月29日）があります。A-8の「こだま」です。
- B-8：残りの幅は、夏（1〜2月）に広く、ほかの季節には狭くなります。一年を1つの幅で測ると、拾う日が夏に集まります。「残りは、同じ幅のiid」という仮定が、実データでは成り立っていません。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `cp_series` | `"Nile"` `"LakeHuron"` `"nhtemp"` |
| `pen_mult` | 0.5〜4 |
| `series` | `"Demand"` `"Temperature"` |
| `year_from` | 2012〜2014 |
| `year_to` | `year_from`以上、2014以下 |
| `width_type` | `"sd"` `"mad"` `"iqr"` |
| `k` | 2〜5 |

読むときの観点：

1. `cp_series`を`"LakeHuron"`にすると、変化点はいくつ見つかるか。B-4のACFと、A-6の結果を合わせて読む。その変化点は、本当に「平均が変わった年」か。
2. `pen_mult`を0.5にすると、`Nile`の変化点はいくつになるか。増えた変化点の区間は、何年続いているか。
3. `width_type`を`"sd"`にすると、拾う日はどれだけ減るか。拾われなくなるのは、どんな日か。
4. `series`を`"Temperature"`にすると、拾った日のうち祝日はいくつあるか。電力需要のときと比べる。
5. 期間を3年分（2012〜2014）にすると、B-8の分散の変化点は、毎年同じ季節に出るか。

正解は1つではありません。自分の設定で出た結果を、最初の設定（Nile、罰則1倍、Demand、2014年、MAD、3倍）の結果と比べてください。